In [1]:
# One Jupyter cell. Put GALA_statistics.xlsx in the notebook folder.
# Uses only the aggregate workbook; no models, GPU, network, or package installs.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter

XLSX = Path('GALA_statistics.xlsx')
OUT = Path('GALA_FIGURES')
DPI = 320
SAVE_PDF = True
if not XLSX.is_file():
    raise FileNotFoundError(f'Put the supplied workbook here: {XLSX.resolve()}')
OUT.mkdir(parents=True, exist_ok=True)
T = pd.read_excel(XLSX, sheet_name=None, engine='openpyxl')
needed = ['Retrieval','Contrasts','Hybrid','Proxy','Diagnostics','Patterns',
          'Lengths','ClassResults','RankCDF','JoinRepair','Experiment']
if missing := sorted(set(needed) - set(T)):
    raise ValueError(f'Wrong/incomplete workbook; missing sheets: {missing}')
R, C, H = T['Retrieval'], T['Contrasts'], T['Hybrid']
P, D, F = T['Proxy'], T['Diagnostics'], T['Patterns']
L, A, K, J = T['Lengths'], T['ClassResults'], T['RankCDF'], T['JoinRepair']
assert not R.duplicated(['target','system']).any()
assert R.groupby('target').n_queries.nunique().eq(1).all()

# Exact primary style constants from fractional-lora-unlearning/FIGURES.
BG, AX_BG, GRID, BLACK = '#FAF8F2', '#F4F1E8', '#D8D3C5', '#15191B'
CM = plt.get_cmap('viridis')
plt.rcParams.update({
    'font.family':'DejaVu Sans', 'font.size':16, 'axes.titlesize':20,
    'axes.labelsize':18, 'xtick.labelsize':15, 'ytick.labelsize':15,
    'legend.fontsize':14, 'figure.titlesize':22, 'axes.linewidth':1.0,
    'lines.linewidth':2.2, 'axes.edgecolor':BLACK, 'text.color':BLACK,
    'axes.labelcolor':BLACK, 'xtick.color':BLACK, 'ytick.color':BLACK,
    'pdf.fonttype':42, 'ps.fonttype':42, 'savefig.pad_inches':0.2})
TARGETS = ['artist_style','style_genre','artist']
TN = {'artist_style':'Artist–style','style_genre':'Style–genre','artist':'Artist'}
PROMPTS = ['full','visual_only','no_style','no_uncertainty','compact']
PN = {'full':'Full','visual_only':'Visual-only','no_style':'No-style',
      'no_uncertainty':'No-uncertainty','compact':'Compact'}
Q, V = 'qwen25vl_3b', 'llava_onevision_0_5b'
SYSTEMS = ['baseline_clip_image'] + [f'{m}__{p}' for m in [Q,V] for p in PROMPTS] + ['baseline_random_fixed']
MODEL_COLORS = {Q:CM(.22), V:CM(.73)}
MARK = dict(zip(PROMPTS,['o','s','^','D','P']))
MADE = []
def name(s):
    if s == 'baseline_clip_image': return 'CLIP image'
    if s == 'baseline_random_fixed': return 'Random'
    m,p = s.split('__'); return ('Qwen ' if m == Q else 'LLaVA ') + PN[p].lower()
def panel(ax, title='', xlabel='', ylabel='', grid='x'):
    ax.set_facecolor(AX_BG); ax.set_axisbelow(True)
    if grid: ax.grid(axis=grid, color=GRID, alpha=.75, lw=.8)
    for spine in ax.spines.values(): spine.set(color=BLACK, linewidth=.9)
    ax.set_title(title, weight='bold', pad=12)
    ax.set_xlabel(xlabel, labelpad=8); ax.set_ylabel(ylabel, labelpad=8)
def save(fig, stem, note=''):
    if note: fig.text(.5,.012,note,ha='center',va='bottom',fontsize=12)
    fig.savefig(OUT/f'{stem}.png',dpi=DPI,bbox_inches='tight',facecolor=BG)
    if SAVE_PDF: fig.savefig(OUT/f'{stem}.pdf',bbox_inches='tight',facecolor=BG)
    MADE.append(stem); plt.close(fig); print(f'[SAVED] {stem}',flush=True)
def gradient_bar(ax,y,left,right,height=.42):
    lo,hi = sorted([left,right])
    if hi <= lo: return
    ax.imshow(np.linspace(.15,.9,256)[None,:],extent=(lo,hi,y-height/2,y+height/2),
              cmap=CM,aspect='auto',alpha=.70,zorder=2,interpolation='bicubic')
    ax.add_patch(Rectangle((lo,y-height/2),hi-lo,height,fill=False,ec=BLACK,lw=.6,zorder=3))
def heat(ax, values, rows, cols, title, colorbar=True, log=False):
    data = np.log1p(values) if log else values
    top = float(np.nanmax(data)) if log else 1
    im = ax.imshow(data,cmap=CM,vmin=0,vmax=max(top,1e-9),aspect='auto')
    panel(ax,title,grid=None)
    ax.set_xticks(range(len(cols)),cols); ax.set_yticks(range(len(rows)),rows)
    for i in range(len(rows)):
        for j in range(len(cols)):
            value=values[i,j]
            rgb=CM(float(data[i,j]/max(top,1e-9)))[:3]
            color='white' if np.dot(rgb,[.2126,.7152,.0722])<.48 else BLACK
            ax.text(j,i,f'{value:.1f}' if log else f'{value:.3f}',ha='center',va='center',color=color,fontsize=12)
    if colorbar: plt.colorbar(im,ax=ax,fraction=.035,pad=.025)
    return im

# 01. All systems and targets, three distinct retrieval measures.
fig,axes=plt.subplots(1,3,figsize=(23,12),facecolor=BG)
for a,metric,title in zip(axes,['hit1','hit10','map10'],['Hit@1','Hit@10','mAP@10']):
    z=R.pivot(index='system',columns='target',values=metric).loc[SYSTEMS,TARGETS]
    heat(a,z.to_numpy(),[name(s) for s in SYSTEMS] if a is axes[0] else ['']*len(SYSTEMS),
         ['Artist–\nstyle','Style–\ngenre','Artist'],title)
fig.suptitle('ID-aligned retrieval atlas',weight='bold',y=.98)
fig.subplots_adjust(left=.17,right=.98,top=.92,bottom=.11,wspace=.13)
save(fig,'01_retrieval_atlas','Shared nonempty cohort; fixed target-specific galleries; query self-matches excluded.')

# 02. Matched Qwen/LLaVA comparisons under every prompt.
fig,axes=plt.subplots(1,3,figsize=(22,8),sharey=True,facecolor=BG)
for a,t in zip(axes,TARGETS):
    s=R[R.target.eq(t)].set_index('system'); panel(a,TN[t],xlabel='Hit@1')
    for y,p in enumerate(PROMPTS):
        q=s.loc[f'{Q}__{p}','hit1']; v=s.loc[f'{V}__{p}','hit1']
        a.plot([q,v],[y,y],c=GRID,lw=5,zorder=1)
        for x,c,m,offset,ha in [(q,CM(.22),'o',-9,'right'),(v,CM(.73),'s',9,'left')]:
            a.scatter(x,y,s=145,c=[c],marker=m,ec=BLACK,lw=.8,zorder=3)
            a.annotate(f'{x:.3f}',(x,y),xytext=(offset,9),textcoords='offset points',ha=ha,fontsize=12)
    a.set_yticks(range(5),[PN[p] for p in PROMPTS]); a.set_ylim(4.7,-.7); a.set_xlim(.24,.52)
fig.legend([Line2D([],[],marker='o',ls='',color=CM(.22)),Line2D([],[],marker='s',ls='',color=CM(.73))],
           ['Qwen2.5-VL-3B','LLaVA-OneVision-0.5B'],loc='lower center',bbox_to_anchor=(.55,.04),ncol=2,frameon=False)
fig.suptitle('Matched-backbone prompt atlas',weight='bold',y=.98)
fig.subplots_adjust(left=.15,right=.98,top=.87,bottom=.20,wspace=.14)
save(fig,'02_matched_prompt_dumbbells')

# 03. Retrieval/proxy frontier. Both axes use the same query population.
fig,axes=plt.subplots(1,3,figsize=(23,8.5),facecolor=BG)
for a,t in zip(axes,TARGETS):
    z=R[R.target.eq(t)].merge(P[P.population.eq(t)],on='system',suffixes=('','_proxy'))
    panel(a,TN[t],xlabel='Regex matches per sentence',ylabel='Hit@1' if a is axes[0] else '')
    for _,r in z.iterrows():
        m,p=r.system.split('__')
        a.scatter(r.mean_proxy,r.hit1,s=3.5*r.mean_words,c=[MODEL_COLORS[m]],marker=MARK[p],alpha=.78,ec=BLACK,lw=.8,zorder=3)
    xy=z[['mean_proxy','hit1']].to_numpy()
    good=np.array([not any((u[0]<=p[0] and u[1]>=p[1]) and np.any(u!=p) for u in xy) for p in xy])
    front=xy[good]; front=front[np.argsort(front[:,0])]
    a.plot(front[:,0],front[:,1],c=BLACK,ls='--',lw=1.2,zorder=2)
    a.margins(.15)
handles=[Line2D([],[],color=MODEL_COLORS[m],marker='o',ls='',markersize=10,label=l)
         for m,l in [(Q,'Qwen'),(V,'LLaVA')]]
handles += [Line2D([],[],color=BLACK,marker=MARK[p],ls='',markersize=9,label=PN[p]) for p in PROMPTS]
fig.legend(handles=handles,loc='lower center',bbox_to_anchor=(.5,.045),ncol=7,frameon=False)
fig.suptitle('Retrieval and the automatic proxy',weight='bold',y=.99)
fig.subplots_adjust(left=.065,right=.985,top=.87,bottom=.23,wspace=.24)
save(fig,'03_retrieval_proxy_frontier','Bubble area: mean description length. Dashed line: observed nondominated points. Proxy is not verified factuality.')

# 04. Image/text mixing paths on the shared cohort.
fig,axes=plt.subplots(1,3,figsize=(22,8),facecolor=BG)
for a,t in zip(axes,TARGETS):
    z=H[H.target.eq(t)].sort_values('alpha_clip'); panel(a,TN[t],xlabel='CLIP weight α',ylabel='Retrieval score' if a is axes[0] else '')
    for metric,color,lab in [('hit1',CM(.25),'Hit@1'),('map10',CM(.73),'mAP@10')]:
        a.plot(z.alpha_clip,z[metric],'-o',c=color,ms=6,label=lab)
        a.fill_between(z.alpha_clip,z[metric],color=color,alpha=.12)
    a.set_xlim(0,1); a.set_ylim(0,1); a.set_xticks([0,.25,.5,.75,1])
fig.legend(*axes[0].get_legend_handles_labels(),loc='lower center',bbox_to_anchor=(.5,.055),ncol=2,frameon=False)
fig.suptitle('Hybrid CLIP–description retrieval paths',weight='bold',y=.98)
fig.subplots_adjust(left=.07,right=.98,top=.86,bottom=.22,wspace=.22)
save(fig,'04_hybrid_paths','α = 0: LLaVA visual-only. α = 1: CLIP. Coefficient sweep is descriptive; no held-out coefficient selection.')

# 05. Smoothed word-count histograms, normalized to unit peak per ridge.
fig,axes=plt.subplots(1,2,figsize=(22,10),sharex=True,facecolor=BG)
maximum=int(L.words.max()); x=np.arange(maximum+1)
kernel=np.exp(-.5*(np.arange(-9,10)/2.5)**2); kernel/=kernel.sum()
for a,m,title in zip(axes,[Q,V],['Qwen2.5-VL-3B','LLaVA-OneVision-0.5B']):
    panel(a,title,xlabel='Words per raw response',grid='x')
    for y,p in enumerate(PROMPTS):
        z=L[L.system.eq(f'{m}__{p}')]; counts=np.zeros_like(x,dtype=float)
        counts[z.words.astype(int)]=z['count']; density=np.convolve(counts/counts.sum(),kernel,mode='same')
        curve=y+.78*density/max(density.max(),1e-12)
        poly=Polygon(np.c_[np.r_[x,x[::-1]],np.r_[curve,np.full(len(x),y)]],closed=True,fc='none',ec='none')
        a.add_patch(poly)
        im=a.imshow(np.linspace(.15,.9,len(x))[None,:],extent=(0,maximum,y,y+.8),cmap=CM,aspect='auto',alpha=.72,zorder=2)
        im.set_clip_path(poly); a.plot(x,curve,c=BLACK,lw=.9,zorder=3)
        median=int(z.sort_values('words').loc[z.sort_values('words')['count'].cumsum().ge(z['count'].sum()/2),'words'].iloc[0])
        a.plot([median,median],[y,y+.82],c=BLACK,ls='--',lw=1,zorder=4)
        a.text(maximum*.98,y+.43,f'median {median}',ha='right',fontsize=12)
    a.set_yticks(np.arange(5)+.18,[PN[p] for p in PROMPTS]); a.set_ylim(-.2,4.95); a.set_xlim(0,maximum)
fig.suptitle('Description-length distributions',weight='bold',y=.98)
fig.subplots_adjust(left=.12,right=.985,top=.88,bottom=.14,wspace=.27)
save(fig,'05_word_length_ridgelines','All 5,000 responses per condition, including empty outputs. Smoothed histograms; each ridge has unit peak.')

# 06. Exact historical regex features; overlapping matches are not atomic claims.
systems=[f'{m}__{p}' for m in [Q,V] for p in PROMPTS]
F=F.copy(); F['per100']=100*F.matches/F.n_outputs
order=F.sort_values('pattern_id').pattern.drop_duplicates().tolist()
z=F.pivot(index='pattern',columns='system',values='per100').loc[order,systems]
fig,a=plt.subplots(figsize=(22,12),facecolor=BG)
im=heat(a,z.to_numpy(),order,[('Q:' if s.startswith(Q) else 'L:')+PN[s.split('__')[1]] for s in systems],
        'Risk-pattern fingerprint',colorbar=False,log=True)
a.tick_params(axis='x',rotation=40)
cb=fig.colorbar(im,ax=a,fraction=.025,pad=.025)
ticks=np.array([0,1,5,10,25,50,100,200]); ticks=ticks[ticks<=z.to_numpy().max()]
cb.set_ticks(np.log1p(ticks)); cb.set_ticklabels([str(t) for t in ticks])
cb.set_label('Matches per 100 responses (log color scale)')
fig.subplots_adjust(left=.22,right=.92,top=.91,bottom=.24)
save(fig,'06_regex_fingerprint','Historical regular expressions; all outputs. A match does not establish an unsupported or false statement.')

# 07. Controlled repair: same retained vectors, different image-to-label join.
fig,axes=plt.subplots(1,2,figsize=(22,8.5),facecolor=BG,gridspec_kw={'width_ratios':[1.2,1]})
a,b=axes; panel(a,'Artist–style: label-join repair',xlabel='Hit@1')
z=J[J.target.eq('artist_style') & J.metric.eq('hit1')].copy()
for y,(_,r) in enumerate(z.iterrows()):
    gradient_bar(a,y,r.position_join,r.id_join)
    a.scatter([r.position_join,r.id_join],[y,y],s=155,c=['white',CM(.72)],ec=BLACK,zorder=4)
    for value,dy in [(r.position_join,-25),(r.id_join,15)]:
        a.annotate(f'{value:.3f}',(value,y),xytext=(0,dy),textcoords='offset points',ha='center',fontsize=14)
a.set_yticks(range(len(z)),[PN[s.split('__')[1]] for s in z.system]); a.set_ylim(len(z)-.35,-.65); a.set_xlim(0,.43)
panel(b,'Empty raw Qwen outputs',xlabel='Fraction of all 5,000 responses')
qd=D[D.model.eq(Q)].set_index('prompt').loc[PROMPTS]
for y,(p,r) in enumerate(qd.iterrows()):
    frac=r.empty_count/r.n_outputs; gradient_bar(b,y,0,frac)
    b.text(frac+.001,y,f'{int(r.empty_count)} ({frac:.2%})',va='center',fontsize=13)
b.set_yticks(range(5),[PN[p] for p in PROMPTS]); b.set_ylim(4.65,-.65); b.set_xlim(0,.093); b.xaxis.set_major_formatter(PercentFormatter(1))
fig.suptitle('Qwen anomaly: an identity-alignment failure',weight='bold',y=.99)
fig.legend([Line2D([],[],marker='o',mfc='white',mec=BLACK,ls='',color=BLACK),
            Line2D([],[],marker='o',mfc=CM(.72),mec=BLACK,ls='',color=CM(.72))],
           ['Positional label join','Identity-based label join'],loc='lower center',bbox_to_anchor=(.48,.055),ncol=2,frameon=False)
fig.subplots_adjust(left=.12,right=.985,top=.84,bottom=.21,wspace=.28)
save(fig,'07_qwen_identity_audit','Left: identical retained embedding rows before/after label remapping. Right: observed empty-output counts; no new generation.')

# 08. Class-level variation, with decoded source artist names and sample sizes.
select=['baseline_clip_image',f'{V}__visual_only',f'{Q}__full',f'{Q}__compact']
ar=A[A.target.eq('artist')]; info=ar[ar.system.eq(select[0])].sort_values('n_queries',ascending=False)
codes=info.class_code.tolist(); names=[f'{r.class_name}  (n={int(r.n_queries)})' for _,r in info.iterrows()]
z=ar.pivot(index='class_code',columns='system',values='hit1').loc[codes,select]
fig,axes=plt.subplots(1,2,figsize=(24,14),facecolor=BG,gridspec_kw={'width_ratios':[1.25,1]})
heat(axes[0],z.to_numpy(),names,['CLIP','LLaVA\nvisual-only','Qwen\nfull','Qwen\ncompact'],'Hit@1 by artist',colorbar=False)
gap=(z[select[1]]-z[select[0]]).to_numpy(); panel(axes[1],'LLaVA visual-only minus CLIP',xlabel='Hit@1 difference')
axes[1].barh(np.arange(len(gap)),gap,color=[CM(.72) if v>=0 else CM(.27) for v in gap],alpha=.75,ec=BLACK,lw=.6)
axes[1].axvline(0,c=BLACK,lw=1); axes[1].set_ylim(len(gap)-.5,-.5); axes[1].set_yticks([])
fig.suptitle('Retrieval heterogeneity across artists',weight='bold',y=.98)
fig.subplots_adjust(left=.27,right=.98,top=.91,bottom=.13,wspace=.14)
save(fig,'08_artist_heterogeneity','Source metadata names, not inferred identities. Unknown Artist is retained as the source category.')

# 09. Paired prompt effects relative to each backbone full prompt.
cp=C[C.metric.eq('hit1') & C.reference.str.endswith('__full')]
order=[f'{m}__{p}' for m in [Q,V] for p in PROMPTS[1:]]
fig,axes=plt.subplots(1,3,figsize=(23,10),sharey=True,facecolor=BG)
for a,t in zip(axes,TARGETS):
    z=cp[cp.target.eq(t)].set_index('system').loc[order]
    panel(a,TN[t],xlabel='Δ Hit@1 relative to own full prompt')
    for y,(_,r) in enumerate(z.iterrows()):
        a.plot([r.artist_ci_low,r.artist_ci_high],[y,y],c=CM(.72),lw=8,alpha=.43,solid_capstyle='round')
        a.plot([r.query_ci_low,r.query_ci_high],[y,y],c=CM(.24),lw=2.5)
        a.scatter(r.delta,y,s=75,c=[CM(.24)],ec=BLACK,lw=.7,zorder=4)
    a.axvline(0,c=BLACK,ls='--',lw=1); a.set_yticks(range(len(order)),[name(s) for s in order]); a.set_ylim(len(order)-.4,-.6)
fig.legend([Line2D([],[],color=CM(.24),lw=2.5),Line2D([],[],color=CM(.72),lw=8,alpha=.43)],
           ['Paired query bootstrap: 95% CI','Paired artist-cluster bootstrap: 95% CI'],loc='lower center',bbox_to_anchor=(.55,.035),ncol=2,frameon=False)
fig.suptitle('Prompt effects with paired uncertainty',weight='bold',y=.98)
fig.subplots_adjust(left=.19,right=.985,top=.88,bottom=.20,wspace=.15)
save(fig,'09_paired_prompt_contrasts','2,000 bootstrap draws; seed 42; fixed gallery. Intervals do not measure generation-to-generation variability.')

# 10. Rank-depth profile from exact top-50 retrieval results.
selected=['baseline_clip_image',f'{V}__visual_only',f'{V}__full',f'{Q}__compact',f'{Q}__full','baseline_random_fixed']
colors=[CM(v) for v in [.08,.76,.60,.43,.27,.91]]
fig,axes=plt.subplots(1,3,figsize=(23,8.5),facecolor=BG)
for a,t in zip(axes,TARGETS):
    panel(a,TN[t],xlabel='Retrieval depth K',ylabel='Queries with a relevant result' if a is axes[0] else '')
    for s,color in zip(selected,colors):
        z=K[K.target.eq(t) & K.system.eq(s)].sort_values('k')
        a.step(z.k,z.hit_rate,where='post',c=color,lw=2.5,label=name(s))
    a.set_xscale('log'); a.set_xticks([1,2,5,10,20,50],['1','2','5','10','20','50']); a.set_xlim(1,50); a.set_ylim(0,1)
    a.yaxis.set_major_formatter(PercentFormatter(1))
fig.legend(*axes[0].get_legend_handles_labels(),loc='lower center',bbox_to_anchor=(.53,.035),ncol=3,frameon=False)
fig.suptitle('Retrieval-depth profiles',weight='bold',y=.99)
fig.subplots_adjust(left=.075,right=.985,top=.86,bottom=.24,wspace=.22)
save(fig,'10_retrieval_depth_profiles','Discrete K = 1…50; same shared cohort. Random control averages three permutations.')

assert len(MADE) == 10
print(f'\nDone: {len(MADE)} figures; PNG'+(' + PDF' if SAVE_PDF else '')+f' in {OUT.resolve()}')


[SAVED] 01_retrieval_atlas
[SAVED] 02_matched_prompt_dumbbells
[SAVED] 03_retrieval_proxy_frontier
[SAVED] 04_hybrid_paths
[SAVED] 05_word_length_ridgelines
[SAVED] 06_regex_fingerprint
[SAVED] 07_qwen_identity_audit
[SAVED] 08_artist_heterogeneity
[SAVED] 09_paired_prompt_contrasts
[SAVED] 10_retrieval_depth_profiles

Done: 10 figures; PNG + PDF in C:\Users\ivan\WORK\workshops\acmMM\GALA\GALA_FIGURES


In [2]:
# GALA v2: boxed numeric annotations and larger fonts.
# Output folder and figure stems are unchanged: existing PNG/PDF files are overwritten.
# One Jupyter cell. Put GALA_statistics.xlsx in the notebook folder.
# Uses only the aggregate workbook; no models, GPU, network, or package installs.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter

XLSX = Path('GALA_statistics.xlsx')
OUT = Path('GALA_FIGURES')
DPI = 320
SAVE_PDF = True
if not XLSX.is_file():
    raise FileNotFoundError(f'Put the supplied workbook here: {XLSX.resolve()}')
OUT.mkdir(parents=True, exist_ok=True)
T = pd.read_excel(XLSX, sheet_name=None, engine='openpyxl')
needed = ['Retrieval','Contrasts','Hybrid','Proxy','Diagnostics','Patterns',
          'Lengths','ClassResults','RankCDF','JoinRepair','Experiment']
if missing := sorted(set(needed) - set(T)):
    raise ValueError(f'Wrong/incomplete workbook; missing sheets: {missing}')
R, C, H = T['Retrieval'], T['Contrasts'], T['Hybrid']
P, D, F = T['Proxy'], T['Diagnostics'], T['Patterns']
L, A, K, J = T['Lengths'], T['ClassResults'], T['RankCDF'], T['JoinRepair']
assert not R.duplicated(['target','system']).any()
assert R.groupby('target').n_queries.nunique().eq(1).all()

# Exact primary style constants from fractional-lora-unlearning/FIGURES.
BG, AX_BG, GRID, BLACK = '#FAF8F2', '#F4F1E8', '#D8D3C5', '#15191B'
CM = plt.get_cmap('viridis')
plt.rcParams.update({
    'font.family':'DejaVu Sans', 'font.size':18, 'axes.titlesize':22,
    'axes.labelsize':20, 'xtick.labelsize':17, 'ytick.labelsize':17,
    'legend.fontsize':16, 'figure.titlesize':24, 'axes.linewidth':1.0,
    'lines.linewidth':2.2, 'axes.edgecolor':BLACK, 'text.color':BLACK,
    'axes.labelcolor':BLACK, 'xtick.color':BLACK, 'ytick.color':BLACK,
    'pdf.fonttype':42, 'ps.fonttype':42, 'savefig.pad_inches':0.2})
TARGETS = ['artist_style','style_genre','artist']
TN = {'artist_style':'Artist–style','style_genre':'Style–genre','artist':'Artist'}
PROMPTS = ['full','visual_only','no_style','no_uncertainty','compact']
PN = {'full':'Full','visual_only':'Visual-only','no_style':'No-style',
      'no_uncertainty':'No-uncertainty','compact':'Compact'}
Q, V = 'qwen25vl_3b', 'llava_onevision_0_5b'
SYSTEMS = ['baseline_clip_image'] + [f'{m}__{p}' for m in [Q,V] for p in PROMPTS] + ['baseline_random_fixed']
MODEL_COLORS = {Q:CM(.22), V:CM(.73)}
MARK = dict(zip(PROMPTS,['o','s','^','D','P']))
# Edit these two settings to adjust every numeric annotation consistently.
NUM_FS = 14
NUM_BBOX = dict(
    boxstyle='round,pad=0.20', facecolor='white',
    edgecolor=BLACK, linewidth=0.7, alpha=0.96
)
MADE = []

def label_box(ax, x, y, text, fontsize=NUM_FS, ha='center',
              va='center', offset=None):
    style = dict(
        fontsize=fontsize, color=BLACK, weight='semibold',
        ha=ha, va=va, bbox=NUM_BBOX.copy(), zorder=10,
        clip_on=False
    )
    if offset is None:
        return ax.text(x, y, text, **style)
    return ax.annotate(
        text, (x, y), xytext=offset,
        textcoords='offset points', **style
    )

def name(s):
    if s == 'baseline_clip_image': return 'CLIP image'
    if s == 'baseline_random_fixed': return 'Random'
    m,p = s.split('__'); return ('Qwen ' if m == Q else 'LLaVA ') + PN[p].lower()
def panel(ax, title='', xlabel='', ylabel='', grid='x'):
    ax.set_facecolor(AX_BG); ax.set_axisbelow(True)
    if grid: ax.grid(axis=grid, color=GRID, alpha=.75, lw=.8)
    for spine in ax.spines.values(): spine.set(color=BLACK, linewidth=.9)
    ax.set_title(title, weight='bold', pad=12)
    ax.set_xlabel(xlabel, labelpad=8); ax.set_ylabel(ylabel, labelpad=8)
def save(fig, stem, note=''):
    if note: fig.text(.5,.012,note,ha='center',va='bottom',fontsize=13)
    fig.savefig(OUT/f'{stem}.png',dpi=DPI,bbox_inches='tight',facecolor=BG)
    if SAVE_PDF: fig.savefig(OUT/f'{stem}.pdf',bbox_inches='tight',facecolor=BG)
    MADE.append(stem); plt.close(fig); print(f'[SAVED] {stem}',flush=True)
def gradient_bar(ax,y,left,right,height=.42):
    lo,hi = sorted([left,right])
    if hi <= lo: return
    ax.imshow(np.linspace(.15,.9,256)[None,:],extent=(lo,hi,y-height/2,y+height/2),
              cmap=CM,aspect='auto',alpha=.70,zorder=2,interpolation='bicubic')
    ax.add_patch(Rectangle((lo,y-height/2),hi-lo,height,fill=False,ec=BLACK,lw=.6,zorder=3))
def heat(ax, values, rows, cols, title, colorbar=True, log=False):
    data = np.log1p(values) if log else values
    top = float(np.nanmax(data)) if log else 1
    im = ax.imshow(data,cmap=CM,vmin=0,vmax=max(top,1e-9),aspect='auto')
    panel(ax,title,grid=None)
    ax.set_xticks(range(len(cols)),cols); ax.set_yticks(range(len(rows)),rows)
    for i in range(len(rows)):
        for j in range(len(cols)):
            value=values[i,j]
            label_box(ax, j, i, f'{value:.1f}' if log else f'{value:.3f}')
    if colorbar: plt.colorbar(im,ax=ax,fraction=.035,pad=.025)
    return im

# 01. All systems and targets, three distinct retrieval measures.
fig,axes=plt.subplots(1,3,figsize=(23,12),facecolor=BG)
for a,metric,title in zip(axes,['hit1','hit10','map10'],['Hit@1','Hit@10','mAP@10']):
    z=R.pivot(index='system',columns='target',values=metric).loc[SYSTEMS,TARGETS]
    heat(a,z.to_numpy(),[name(s) for s in SYSTEMS] if a is axes[0] else ['']*len(SYSTEMS),
         ['Artist–\nstyle','Style–\ngenre','Artist'],title)
fig.suptitle('ID-aligned retrieval atlas',weight='bold',y=.98)
fig.subplots_adjust(left=.17,right=.98,top=.92,bottom=.11,wspace=.13)
save(fig,'01_retrieval_atlas','Shared nonempty cohort; fixed target-specific galleries; query self-matches excluded.')

# 02. Matched Qwen/LLaVA comparisons under every prompt.
fig,axes=plt.subplots(1,3,figsize=(22,8),sharey=True,facecolor=BG)
for a,t in zip(axes,TARGETS):
    s=R[R.target.eq(t)].set_index('system'); panel(a,TN[t],xlabel='Hit@1')
    for y,p in enumerate(PROMPTS):
        q=s.loc[f'{Q}__{p}','hit1']; v=s.loc[f'{V}__{p}','hit1']
        a.plot([q,v],[y,y],c=GRID,lw=5,zorder=1)
        for x,c,m,offset,ha in [(q,CM(.22),'o',-9,'right'),(v,CM(.73),'s',9,'left')]:
            a.scatter(x,y,s=145,c=[c],marker=m,ec=BLACK,lw=.8,zorder=3)
            label_box(a, x, y, f'{x:.3f}', ha=ha, offset=(offset,18))
    a.set_yticks(range(5),[PN[p] for p in PROMPTS]); a.set_ylim(4.7,-.7); a.set_xlim(.24,.52)
fig.legend([Line2D([],[],marker='o',ls='',color=CM(.22)),Line2D([],[],marker='s',ls='',color=CM(.73))],
           ['Qwen2.5-VL-3B','LLaVA-OneVision-0.5B'],loc='lower center',bbox_to_anchor=(.55,.04),ncol=2,frameon=False)
fig.suptitle('Matched-backbone prompt atlas',weight='bold',y=.98)
fig.subplots_adjust(left=.15,right=.98,top=.87,bottom=.20,wspace=.14)
save(fig,'02_matched_prompt_dumbbells')

# 03. Retrieval/proxy frontier. Both axes use the same query population.
fig,axes=plt.subplots(1,3,figsize=(23,8.5),facecolor=BG)
for a,t in zip(axes,TARGETS):
    z=R[R.target.eq(t)].merge(P[P.population.eq(t)],on='system',suffixes=('','_proxy'))
    panel(a,TN[t],xlabel='Regex matches per sentence',ylabel='Hit@1' if a is axes[0] else '')
    for _,r in z.iterrows():
        m,p=r.system.split('__')
        a.scatter(r.mean_proxy,r.hit1,s=3.5*r.mean_words,c=[MODEL_COLORS[m]],marker=MARK[p],alpha=.78,ec=BLACK,lw=.8,zorder=3)
    xy=z[['mean_proxy','hit1']].to_numpy()
    good=np.array([not any((u[0]<=p[0] and u[1]>=p[1]) and np.any(u!=p) for u in xy) for p in xy])
    front=xy[good]; front=front[np.argsort(front[:,0])]
    a.plot(front[:,0],front[:,1],c=BLACK,ls='--',lw=1.2,zorder=2)
    a.margins(.15)
handles=[Line2D([],[],color=MODEL_COLORS[m],marker='o',ls='',markersize=10,label=l)
         for m,l in [(Q,'Qwen'),(V,'LLaVA')]]
handles += [Line2D([],[],color=BLACK,marker=MARK[p],ls='',markersize=9,label=PN[p]) for p in PROMPTS]
fig.legend(handles=handles,loc='lower center',bbox_to_anchor=(.5,.045),ncol=7,frameon=False)
fig.suptitle('Retrieval and the automatic proxy',weight='bold',y=.99)
fig.subplots_adjust(left=.065,right=.985,top=.87,bottom=.23,wspace=.24)
save(fig,'03_retrieval_proxy_frontier','Bubble area: mean description length. Dashed line: observed nondominated points. Proxy is not verified factuality.')

# 04. Image/text mixing paths on the shared cohort.
fig,axes=plt.subplots(1,3,figsize=(22,8),facecolor=BG)
for a,t in zip(axes,TARGETS):
    z=H[H.target.eq(t)].sort_values('alpha_clip'); panel(a,TN[t],xlabel='CLIP weight α',ylabel='Retrieval score' if a is axes[0] else '')
    for metric,color,lab in [('hit1',CM(.25),'Hit@1'),('map10',CM(.73),'mAP@10')]:
        a.plot(z.alpha_clip,z[metric],'-o',c=color,ms=6,label=lab)
        a.fill_between(z.alpha_clip,z[metric],color=color,alpha=.12)
    a.set_xlim(0,1); a.set_ylim(0,1); a.set_xticks([0,.25,.5,.75,1])
fig.legend(*axes[0].get_legend_handles_labels(),loc='lower center',bbox_to_anchor=(.5,.055),ncol=2,frameon=False)
fig.suptitle('Hybrid CLIP–description retrieval paths',weight='bold',y=.98)
fig.subplots_adjust(left=.07,right=.98,top=.86,bottom=.22,wspace=.22)
save(fig,'04_hybrid_paths','α = 0: LLaVA visual-only. α = 1: CLIP. Coefficient sweep is descriptive; no held-out coefficient selection.')

# 05. Smoothed word-count histograms, normalized to unit peak per ridge.
fig,axes=plt.subplots(1,2,figsize=(22,10),sharex=True,facecolor=BG)
maximum=int(L.words.max()); x=np.arange(maximum+1)
kernel=np.exp(-.5*(np.arange(-9,10)/2.5)**2); kernel/=kernel.sum()
for a,m,title in zip(axes,[Q,V],['Qwen2.5-VL-3B','LLaVA-OneVision-0.5B']):
    panel(a,title,xlabel='Words per raw response',grid='x')
    for y,p in enumerate(PROMPTS):
        z=L[L.system.eq(f'{m}__{p}')]; counts=np.zeros_like(x,dtype=float)
        counts[z.words.astype(int)]=z['count']; density=np.convolve(counts/counts.sum(),kernel,mode='same')
        curve=y+.78*density/max(density.max(),1e-12)
        poly=Polygon(np.c_[np.r_[x,x[::-1]],np.r_[curve,np.full(len(x),y)]],closed=True,fc='none',ec='none')
        a.add_patch(poly)
        im=a.imshow(np.linspace(.15,.9,len(x))[None,:],extent=(0,maximum,y,y+.8),cmap=CM,aspect='auto',alpha=.72,zorder=2)
        im.set_clip_path(poly); a.plot(x,curve,c=BLACK,lw=.9,zorder=3)
        median=int(z.sort_values('words').loc[z.sort_values('words')['count'].cumsum().ge(z['count'].sum()/2),'words'].iloc[0])
        a.plot([median,median],[y,y+.82],c=BLACK,ls='--',lw=1,zorder=4)
        label_box(a, maximum*.98, y+.43, f'median {median}', ha='right')
    a.set_yticks(np.arange(5)+.18,[PN[p] for p in PROMPTS]); a.set_ylim(-.2,4.95); a.set_xlim(0,maximum)
fig.suptitle('Description-length distributions',weight='bold',y=.98)
fig.subplots_adjust(left=.12,right=.985,top=.88,bottom=.14,wspace=.27)
save(fig,'05_word_length_ridgelines','All 5,000 responses per condition, including empty outputs. Smoothed histograms; each ridge has unit peak.')

# 06. Exact historical regex features; overlapping matches are not atomic claims.
systems=[f'{m}__{p}' for m in [Q,V] for p in PROMPTS]
F=F.copy(); F['per100']=100*F.matches/F.n_outputs
order=F.sort_values('pattern_id').pattern.drop_duplicates().tolist()
z=F.pivot(index='pattern',columns='system',values='per100').loc[order,systems]
fig,a=plt.subplots(figsize=(22,12),facecolor=BG)
im=heat(a,z.to_numpy(),order,[('Q:' if s.startswith(Q) else 'L:')+PN[s.split('__')[1]] for s in systems],
        'Risk-pattern fingerprint',colorbar=False,log=True)
a.tick_params(axis='x',rotation=40)
cb=fig.colorbar(im,ax=a,fraction=.025,pad=.025)
ticks=np.array([0,1,5,10,25,50,100,200]); ticks=ticks[ticks<=z.to_numpy().max()]
cb.set_ticks(np.log1p(ticks)); cb.set_ticklabels([str(t) for t in ticks])
cb.set_label('Matches per 100 responses (log color scale)')
fig.subplots_adjust(left=.22,right=.92,top=.91,bottom=.24)
save(fig,'06_regex_fingerprint','Historical regular expressions; all outputs. A match does not establish an unsupported or false statement.')

# 07. Controlled repair: same retained vectors, different image-to-label join.
fig,axes=plt.subplots(1,2,figsize=(22,8.5),facecolor=BG,gridspec_kw={'width_ratios':[1.2,1]})
a,b=axes; panel(a,'Artist–style: label-join repair',xlabel='Hit@1')
z=J[J.target.eq('artist_style') & J.metric.eq('hit1')].copy()
for y,(_,r) in enumerate(z.iterrows()):
    gradient_bar(a,y,r.position_join,r.id_join)
    a.scatter([r.position_join,r.id_join],[y,y],s=155,c=['white',CM(.72)],ec=BLACK,zorder=4)
    for value,dy in [(r.position_join,-28),(r.id_join,24)]:
        label_box(a, value, y, f'{value:.3f}', fontsize=16, offset=(0,dy))
a.set_yticks(range(len(z)),[PN[s.split('__')[1]] for s in z.system]); a.set_ylim(len(z)-.35,-.65); a.set_xlim(0,.43)
panel(b,'Empty raw Qwen outputs',xlabel='Fraction of all 5,000 responses')
qd=D[D.model.eq(Q)].set_index('prompt').loc[PROMPTS]
for y,(p,r) in enumerate(qd.iterrows()):
    frac=r.empty_count/r.n_outputs; gradient_bar(b,y,0,frac)
    label_box(b, frac+.001, y, f'{int(r.empty_count)} ({frac:.2%})', ha='left', fontsize=15)
b.set_yticks(range(5),[PN[p] for p in PROMPTS]); b.set_ylim(4.65,-.65); b.set_xlim(0,.093); b.xaxis.set_major_formatter(PercentFormatter(1))
fig.suptitle('Qwen anomaly: an identity-alignment failure',weight='bold',y=.99)
fig.legend([Line2D([],[],marker='o',mfc='white',mec=BLACK,ls='',color=BLACK),
            Line2D([],[],marker='o',mfc=CM(.72),mec=BLACK,ls='',color=CM(.72))],
           ['Positional label join','Identity-based label join'],loc='lower center',bbox_to_anchor=(.48,.055),ncol=2,frameon=False)
fig.subplots_adjust(left=.12,right=.985,top=.84,bottom=.21,wspace=.28)
save(fig,'07_qwen_identity_audit','Left: identical retained embedding rows before/after label remapping. Right: observed empty-output counts; no new generation.')

# 08. Class-level variation, with decoded source artist names and sample sizes.
select=['baseline_clip_image',f'{V}__visual_only',f'{Q}__full',f'{Q}__compact']
ar=A[A.target.eq('artist')]; info=ar[ar.system.eq(select[0])].sort_values('n_queries',ascending=False)
codes=info.class_code.tolist(); names=[f'{r.class_name}  (n={int(r.n_queries)})' for _,r in info.iterrows()]
z=ar.pivot(index='class_code',columns='system',values='hit1').loc[codes,select]
fig,axes=plt.subplots(1,2,figsize=(24,14),facecolor=BG,gridspec_kw={'width_ratios':[1.25,1]})
heat(axes[0],z.to_numpy(),names,['CLIP','LLaVA\nvisual-only','Qwen\nfull','Qwen\ncompact'],'Hit@1 by artist',colorbar=False)
gap=(z[select[1]]-z[select[0]]).to_numpy(); panel(axes[1],'LLaVA visual-only minus CLIP',xlabel='Hit@1 difference')
axes[1].barh(np.arange(len(gap)),gap,color=[CM(.72) if v>=0 else CM(.27) for v in gap],alpha=.75,ec=BLACK,lw=.6)
axes[1].axvline(0,c=BLACK,lw=1); axes[1].set_ylim(len(gap)-.5,-.5); axes[1].set_yticks([])
fig.suptitle('Retrieval heterogeneity across artists',weight='bold',y=.98)
fig.subplots_adjust(left=.27,right=.98,top=.91,bottom=.13,wspace=.14)
save(fig,'08_artist_heterogeneity','Source metadata names, not inferred identities. Unknown Artist is retained as the source category.')

# 09. Paired prompt effects relative to each backbone full prompt.
cp=C[C.metric.eq('hit1') & C.reference.str.endswith('__full')]
order=[f'{m}__{p}' for m in [Q,V] for p in PROMPTS[1:]]
fig,axes=plt.subplots(1,3,figsize=(23,10),sharey=True,facecolor=BG)
for a,t in zip(axes,TARGETS):
    z=cp[cp.target.eq(t)].set_index('system').loc[order]
    panel(a,TN[t],xlabel='Δ Hit@1 relative to own full prompt')
    for y,(_,r) in enumerate(z.iterrows()):
        a.plot([r.artist_ci_low,r.artist_ci_high],[y,y],c=CM(.72),lw=8,alpha=.43,solid_capstyle='round')
        a.plot([r.query_ci_low,r.query_ci_high],[y,y],c=CM(.24),lw=2.5)
        a.scatter(r.delta,y,s=75,c=[CM(.24)],ec=BLACK,lw=.7,zorder=4)
    a.axvline(0,c=BLACK,ls='--',lw=1); a.set_yticks(range(len(order)),[name(s) for s in order]); a.set_ylim(len(order)-.4,-.6)
fig.legend([Line2D([],[],color=CM(.24),lw=2.5),Line2D([],[],color=CM(.72),lw=8,alpha=.43)],
           ['Paired query bootstrap: 95% CI','Paired artist-cluster bootstrap: 95% CI'],loc='lower center',bbox_to_anchor=(.55,.035),ncol=2,frameon=False)
fig.suptitle('Prompt effects with paired uncertainty',weight='bold',y=.98)
fig.subplots_adjust(left=.19,right=.985,top=.88,bottom=.20,wspace=.15)
save(fig,'09_paired_prompt_contrasts','2,000 bootstrap draws; seed 42; fixed gallery. Intervals do not measure generation-to-generation variability.')

# 10. Rank-depth profile from exact top-50 retrieval results.
selected=['baseline_clip_image',f'{V}__visual_only',f'{V}__full',f'{Q}__compact',f'{Q}__full','baseline_random_fixed']
colors=[CM(v) for v in [.08,.76,.60,.43,.27,.91]]
fig,axes=plt.subplots(1,3,figsize=(23,8.5),facecolor=BG)
for a,t in zip(axes,TARGETS):
    panel(a,TN[t],xlabel='Retrieval depth K',ylabel='Queries with a relevant result' if a is axes[0] else '')
    for s,color in zip(selected,colors):
        z=K[K.target.eq(t) & K.system.eq(s)].sort_values('k')
        a.step(z.k,z.hit_rate,where='post',c=color,lw=2.5,label=name(s))
    a.set_xscale('log'); a.set_xticks([1,2,5,10,20,50],['1','2','5','10','20','50']); a.set_xlim(1,50); a.set_ylim(0,1)
    a.yaxis.set_major_formatter(PercentFormatter(1))
fig.legend(*axes[0].get_legend_handles_labels(),loc='lower center',bbox_to_anchor=(.53,.035),ncol=3,frameon=False)
fig.suptitle('Retrieval-depth profiles',weight='bold',y=.99)
fig.subplots_adjust(left=.075,right=.985,top=.86,bottom=.24,wspace=.22)
save(fig,'10_retrieval_depth_profiles','Discrete K = 1…50; same shared cohort. Random control averages three permutations.')

assert len(MADE) == 10
print(f'\nDone: {len(MADE)} figures; PNG'+(' + PDF' if SAVE_PDF else '')+f' in {OUT.resolve()}')

[SAVED] 01_retrieval_atlas
[SAVED] 02_matched_prompt_dumbbells
[SAVED] 03_retrieval_proxy_frontier
[SAVED] 04_hybrid_paths
[SAVED] 05_word_length_ridgelines
[SAVED] 06_regex_fingerprint
[SAVED] 07_qwen_identity_audit
[SAVED] 08_artist_heterogeneity
[SAVED] 09_paired_prompt_contrasts
[SAVED] 10_retrieval_depth_profiles

Done: 10 figures; PNG + PDF in C:\Users\ivan\WORK\workshops\acmMM\GALA\GALA_FIGURES
